This file contains my solution to a task from [Data Science Simulator](https://karpov.courses/simulator-ds), a hands-on training course in data analysis and machine learning by karpov.courses.

This file is shared with the permission of the course authors, and in a reduced form: the original problem statements and datasets are omitted. What's kept is my own code, reasoning, and commentary.

---

# nDCG — ranking quality metrics

Implementing the family of ranking-quality metrics step by step: Cumulative Gain (CG), Discounted Cumulative Gain (DCG), its normalized version (NDCG), and the average NDCG across many queries. Each function takes a list of relevance labels already ordered as the model returned them, and a cutoff `k`.

Reference: [Discounted cumulative gain (Wikipedia)](https://en.wikipedia.org/wiki/Discounted_cumulative_gain).

## 1. CG@k — Cumulative Gain

**CG@k** simply sums the relevance of the top-`k` items. It ignores the order within the top-`k`, which is its main limitation: reordering the results does not change the score.

In [ ]:
from typing import List

import numpy as np

def cumulative_gain(relevance: List[float], k: int) -> float:
    """Score is cumulative gain at k (CG@k)

    Parameters
    ----------
    relevance:  `List[float]`
        Relevance labels (Ranks)
    k : `int`
        Number of elements to be counted

    Returns
    -------
    score : float
    """

    score = sum(relevance[:k])

    return score

## 2. DCG@k — Discounted Cumulative Gain

DCG fixes CG's blind spot by discounting items further down the list.

**Why a logarithm in the denominator?** A logarithmic discount makes the penalty grow quickly at first and then level off, which matches how users actually scan a results list:
- Between positions 1 and 2 the difference in attention is huge — almost everyone looks at the first result, noticeably fewer at the second.
- Between positions 9 and 10 the difference is tiny — whoever scrolled to the ninth will almost certainly see the tenth too.

**`standard` vs `industry`.** The `industry` variant penalizes irrelevant documents ranked above relevant ones more heavily — useful in domains where users really need the correct answer as high as possible. The penalty still comes from the logarithm in the denominator, but because the numerator is `2**rel_i - 1` instead of plain `rel_i`, the effect is stronger than in standard DCG.

In [ ]:
from typing import List

import numpy as np

def discounted_cumulative_gain(relevance: List[float], k: int, method: str = "standard") -> float:
    """Discounted Cumulative Gain

    Parameters
    ----------
    relevance : `List[float]`
        Video relevance list
    k : `int`
        Count relevance to compute
    method : `str`, optional
        Metric implementation method, takes the values
        `standard` - adds weight to the denominator
        `industry` - adds weights to the numerator and denominator
        `raise ValueError` - for any value

    Returns
    -------
    score : `float`
        Metric score
    """

    if method == "standard":
        score = sum(j/np.log2(i+2) for i, j in enumerate(relevance[:k]))
    elif method == "industry":
        score = sum((2**j-1)/np.log2(i+2) for i, j in enumerate(relevance[:k]))
    else:
        raise ValueError

    return score

## 3. NDCG@k — Normalized DCG

DCG values are not comparable across queries, since longer or more-relevant result sets accumulate larger scores. NDCG fixes this by normalizing.

**Normalization (IDCG).** NDCG divides DCG by the *ideal* DCG (IDCG) — the score you would get if all relevant documents were both present in the output and ordered by descending relevance. Note that the ideal ranking may even contain a different set of documents than the actual output, not just a different order. Dividing by IDCG brings the metric into the `[0, 1]` range and makes it comparable across queries.

In [ ]:
from typing import List

import numpy as np

def normalized_dcg(relevance: List[float], k: int, method: str = "standard") -> float:
    """Normalized Discounted Cumulative Gain

    Parameters
    ----------
    relevance : `List[float]`
        Video relevance list
    k : `int`
        Count relevance to compute
    method : `str`, optional
        Metric implementation method, takes the values
        `standard` - adds weight to the denominator
        `industry` - adds weights to the numerator and denominator
        `raise ValueError` - for any value

    Returns
    -------
    score : `float`
        Metric score
    """
    
    relevance_sorted = sorted(relevance, reverse=True)

    if method == "standard":
        score = sum(j/np.log2(i+2) for i, j in enumerate(relevance[:k]))
        ideal_score = sum(j/np.log2(i+2) for i, j in enumerate(relevance_sorted[:k]))
    elif method == "industry":
        score = sum((2**j-1)/np.log2(i+2) for i, j in enumerate(relevance[:k]))
        ideal_score = sum((2**j-1)/np.log2(i+2) for i, j in enumerate(relevance_sorted[:k]))
    else:
        raise ValueError

    return score / ideal_score

## 4. Average NDCG@k

**Average NDCG@k** averages NDCG over many queries to produce a single quality number for the ranking system as a whole.

In [ ]:
from typing import List

import numpy as np

def avg_ndcg(list_relevances: List[List[float]], k: int, method: str = "standard") -> float:
    """average Normalized Discounted Cumulative Gain

    Parameters
    ----------
    list_relevances : `List[List[float]]`
        Video relevance matrix for various queries
    k : `int`
        Count relevance to compute
    method : `str`, optional
        Metric implementation method, takes the values
        `standard` - adds weight to the denominator
        `industry` - adds weights to the numerator and denominator
        `raise ValueError` - for any value

    Returns
    -------
    score : `float`
        Metric score
    """
    
    relevance_queries = []

    for relevance in list_relevances:
        relevance_queries.append(normalized_dcg(relevance, k, method))

    return np.mean(relevance_queries)
